In [ ]:
# ============================================================
# NG05-1 — Setup and Single Fold/K Classification Validation
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 80)
print("NG05-1 — CLASSIFICATION DATA VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval"
)

FOLD_DIR = BASE_DIR / "folds"
VECTOR_DIR = BASE_DIR / "topic_vectors"
MODEL_DIR = BASE_DIR / "models"

# ------------------------------------------------------------
# Validate one representative experiment
# ------------------------------------------------------------

fold = 1
topics = 2

train_file = (
    FOLD_DIR
    / f"Fold_{fold}"
    / "train.xlsx"
)

test_file = (
    FOLD_DIR
    / f"Fold_{fold}"
    / "test.xlsx"
)

train_vector_file = (
    VECTOR_DIR
    / f"Fold_{fold}"
    / f"K_{topics}"
    / "train_topic_vectors.npy"
)

test_vector_file = (
    VECTOR_DIR
    / f"Fold_{fold}"
    / f"K_{topics}"
    / "test_topic_vectors.npy"
)

# ------------------------------------------------------------
# Load fold data
# ------------------------------------------------------------

train_df = pd.read_excel(
    train_file
)

test_df = pd.read_excel(
    test_file
)

# ------------------------------------------------------------
# Load topic vectors
# ------------------------------------------------------------

X_train = np.load(
    train_vector_file
)

X_test = np.load(
    test_vector_file
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

assert "LabelID" in train_df.columns
assert "LabelID" in test_df.columns

y_train = (
    train_df["LabelID"]
    .to_numpy()
)

y_test = (
    test_df["LabelID"]
    .to_numpy()
)

# ------------------------------------------------------------
# Structural checks
# ------------------------------------------------------------

print("\nFold                 :", fold)
print("Topics               :", topics)

print("\nTrain dataframe      :", train_df.shape)
print("Test dataframe       :", test_df.shape)

print("Train topic vectors  :", X_train.shape)
print("Test topic vectors   :", X_test.shape)

print("Train labels         :", y_train.shape)
print("Test labels          :", y_test.shape)

print(
    "Train classes        :",
    len(np.unique(y_train))
)

print(
    "Test classes         :",
    len(np.unique(y_test))
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert X_train.ndim == 2
assert X_test.ndim == 2

assert X_train.shape[0] == len(train_df)
assert X_test.shape[0] == len(test_df)

assert X_train.shape[0] == len(y_train)
assert X_test.shape[0] == len(y_test)

assert X_train.shape[1] == topics
assert X_test.shape[1] == topics

assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()

assert not pd.isna(y_train).any()
assert not pd.isna(y_test).any()

# ------------------------------------------------------------
# LDA topic-distribution checks
# ------------------------------------------------------------

train_row_sums = X_train.sum(
    axis=1
)

test_row_sums = X_test.sum(
    axis=1
)

print(
    "\nTrain vector row-sum range:",
    train_row_sums.min(),
    "to",
    train_row_sums.max()
)

print(
    "Test vector row-sum range :",
    test_row_sums.min(),
    "to",
    test_row_sums.max()
)

assert np.allclose(
    train_row_sums,
    1.0,
    atol=1e-5
)

assert np.allclose(
    test_row_sums,
    1.0,
    atol=1e-5
)

assert (X_train >= 0).all()
assert (X_test >= 0).all()

# ------------------------------------------------------------
# Label distributions
# ------------------------------------------------------------

train_label_counts = (
    train_df["LabelID"]
    .value_counts()
    .sort_index()
)

test_label_counts = (
    test_df["LabelID"]
    .value_counts()
    .sort_index()
)

label_summary = pd.DataFrame({
    "TrainDocuments":
        train_label_counts,

    "TestDocuments":
        test_label_counts
}).fillna(0).astype(int)

print("\n" + "=" * 80)
print("LABEL DISTRIBUTION")
print("=" * 80)

display(label_summary)

print("\n" + "=" * 80)
print("✓ TOPIC VECTOR DIMENSIONS MATCH FOLD DATA")
print("✓ LABEL COUNTS MATCH TOPIC VECTOR ROW COUNTS")
print("✓ TOPIC VECTOR DIMENSION MATCHES K")
print("✓ TOPIC DISTRIBUTIONS SUM TO 1")
print("✓ NO NON-FINITE VALUES")
print("=" * 80)

Mounted at /content/drive
NG05-1 — CLASSIFICATION DATA VALIDATION

Fold                 : 1
Topics               : 2

Train dataframe      : (14637, 7)
Test dataframe       : (3660, 7)
Train topic vectors  : (14637, 2)
Test topic vectors   : (3660, 2)
Train labels         : (14637,)
Test labels          : (3660,)
Train classes        : 20
Test classes         : 20

Train vector row-sum range: 0.99999994 to 1.0000001
Test vector row-sum range : 0.99999994 to 1.0000001

LABEL DISTRIBUTION


,TrainDocuments,TestDocuments
LabelID,,
0,623,156
1,763,190
2,757,189
3,771,192
4,742,185
5,784,196
6,767,192
7,747,187
8,774,194



✓ TOPIC VECTOR DIMENSIONS MATCH FOLD DATA
✓ LABEL COUNTS MATCH TOPIC VECTOR ROW COUNTS
✓ TOPIC VECTOR DIMENSION MATCHES K
✓ TOPIC DISTRIBUTIONS SUM TO 1
✓ NO NON-FINITE VALUES


In [ ]:
# ============================================================
# NG05-2 — Original Classification Protocol
#          + Single Logistic Regression Validation
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    matthews_corrcoef
)

warnings.filterwarnings("ignore")

print("=" * 80)
print("NG05-2 — ORIGINAL CLASSIFICATION PROTOCOL")
print("=" * 80)

# ============================================================
# Configuration
# ============================================================

CONFIG_FILE = (
    BASE_DIR
    / "config"
    / "config.json"
)

with open(CONFIG_FILE, "r") as f:
    CONFIG = json.load(f)

N_FOLDS = CONFIG["n_folds"]
SEED = CONFIG["random_seed"]

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("\nProject :", CONFIG["project_name"])
print("Folds   :", N_FOLDS)
print("Seed    :", SEED)

# ============================================================
# Classification output directory
# ============================================================

CLASSIFICATION_DIR = (
    BASE_DIR
    / "classification"
)

CLASSIFICATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# Generic topic-vector loader
# ============================================================

def load_theta_dataset(fold, topics):

    train_df = pd.read_excel(
        FOLD_DIR
        / f"Fold_{fold}"
        / "train.xlsx"
    )

    test_df = pd.read_excel(
        FOLD_DIR
        / f"Fold_{fold}"
        / "test.xlsx"
    )

    X_train = np.load(
        VECTOR_DIR
        / f"Fold_{fold}"
        / f"K_{topics}"
        / "train_topic_vectors.npy"
    )

    X_test = np.load(
        VECTOR_DIR
        / f"Fold_{fold}"
        / f"K_{topics}"
        / "test_topic_vectors.npy"
    )

    y_train = (
        train_df["LabelID"]
        .to_numpy()
    )

    y_test = (
        test_df["LabelID"]
        .to_numpy()
    )

    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    assert X_train.shape[0] == len(y_train)
    assert X_test.shape[0] == len(y_test)

    assert X_train.shape[1] == topics
    assert X_test.shape[1] == topics

    assert np.isfinite(X_train).all()
    assert np.isfinite(X_test).all()

    return (
        X_train,
        X_test,
        y_train,
        y_test
    )


# ============================================================
# Logistic Regression
# ============================================================
#
# Single-model validation first.
# Full 495-model evaluation comes only after this passes.
# ============================================================

TEST_FOLD = 1
TEST_K = 2

X_train, X_test, y_train, y_test = (
    load_theta_dataset(
        TEST_FOLD,
        TEST_K
    )
)

clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=SEED
)

clf.fit(
    X_train,
    y_train
)

y_pred = clf.predict(
    X_test
)

# ============================================================
# Original downstream metrics
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

balanced_accuracy = (
    balanced_accuracy_score(
        y_test,
        y_pred
    )
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )
)

mcc = matthews_corrcoef(
    y_test,
    y_pred
)

# ============================================================
# Result
# ============================================================

single_lr_result = pd.DataFrame(
    [{
        "ExperimentID":
            f"NG_F{TEST_FOLD}_K{TEST_K}",

        "Fold":
            TEST_FOLD,

        "Topics":
            TEST_K,

        "Accuracy":
            accuracy,

        "BalancedAccuracy":
            balanced_accuracy,

        "Precision":
            precision,

        "Recall":
            recall,

        "F1":
            f1,

        "MCC":
            mcc
    }]
)

print("\n" + "=" * 80)
print("SINGLE-EXPERIMENT LOGISTIC REGRESSION RESULT")
print("=" * 80)

display(
    single_lr_result
)

# ============================================================
# Validation
# ============================================================

metric_columns = [
    "Accuracy",
    "BalancedAccuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC"
]

assert np.isfinite(
    single_lr_result[
        metric_columns
    ].to_numpy()
).all()

assert 0 <= accuracy <= 1
assert 0 <= balanced_accuracy <= 1
assert 0 <= precision <= 1
assert 0 <= recall <= 1
assert 0 <= f1 <= 1
assert -1 <= mcc <= 1

print("\nTraining documents :", len(y_train))
print("Test documents     :", len(y_test))
print("Number of topics   :", X_train.shape[1])
print("Number of classes  :", len(np.unique(y_train)))

print("\n" + "=" * 80)
print("✓ TOPIC VECTORS LOADED SUCCESSFULLY")
print("✓ LOGISTIC REGRESSION TRAINED SUCCESSFULLY")
print("✓ PREDICTIONS GENERATED SUCCESSFULLY")
print("✓ ALL DOWNSTREAM METRICS ARE VALID")
print("=" * 80)

NG05-2 — ORIGINAL CLASSIFICATION PROTOCOL

Project : NewsGroupsTopicEval
Folds   : 5
Seed    : 42

SINGLE-EXPERIMENT LOGISTIC REGRESSION RESULT


,ExperimentID,Fold,Topics,Accuracy,BalancedAccuracy,Precision,Recall,F1,MCC
0,NG_F1_K2,1,2,0.06776,0.064125,0.062948,0.064125,0.030098,0.020739



Training documents : 14637
Test documents     : 3660
Number of topics   : 2
Number of classes  : 20

✓ TOPIC VECTORS LOADED SUCCESSFULLY
✓ LOGISTIC REGRESSION TRAINED SUCCESSFULLY
✓ PREDICTIONS GENERATED SUCCESSFULLY
✓ ALL DOWNSTREAM METRICS ARE VALID


In [ ]:
# ============================================================
# NG05-3 — Logistic Regression over All 495 Experiments
#          with Checkpoint / Resume Support
# ============================================================

import gc
import time

from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    matthews_corrcoef
)

print("=" * 80)
print("NG05-3 — LOGISTIC REGRESSION: ALL 495 EXPERIMENTS")
print("=" * 80)

# ============================================================
# Output files
# ============================================================

LR_CHECKPOINT_FILE = (
    CLASSIFICATION_DIR
    / "ng05_logistic_regression_checkpoint.csv"
)

LR_FINAL_FILE = (
    CLASSIFICATION_DIR
    / "ng05_logistic_regression_all_models.csv"
)

# ============================================================
# Logistic Regression configuration
# ============================================================

LR_MAX_ITER = 1000
LR_CLASS_WEIGHT = "balanced"

print("\nLogistic Regression Configuration")
print("max_iter     :", LR_MAX_ITER)
print("class_weight :", LR_CLASS_WEIGHT)
print("random_seed  :", SEED)

# ============================================================
# Load existing checkpoint, if available
# ============================================================

if LR_CHECKPOINT_FILE.exists():

    checkpoint_df = pd.read_csv(
        LR_CHECKPOINT_FILE
    )

    results = checkpoint_df.to_dict(
        "records"
    )

    completed = set(
        zip(
            checkpoint_df["Fold"].astype(int),
            checkpoint_df["Topics"].astype(int)
        )
    )

    print(
        "\nCheckpoint found."
    )

    print(
        "Completed experiments:",
        len(completed)
    )

else:

    results = []
    completed = set()

    print(
        "\nNo checkpoint found."
    )

    print(
        "Starting from the beginning."
    )

# ============================================================
# Experiments
# ============================================================

experiments = [
    (fold, topics)
    for fold in range(1, N_FOLDS + 1)
    for topics in range(2, 101)
]

assert len(experiments) == 495

remaining = [
    experiment
    for experiment in experiments
    if experiment not in completed
]

print(
    "Remaining experiments:",
    len(remaining)
)

# ============================================================
# Evaluate
# ============================================================

for fold, topics in tqdm(
    remaining,
    desc="Logistic Regression"
):

    start_time = time.time()

    # --------------------------------------------------------
    # Load LDA topic representations and labels
    # --------------------------------------------------------

    X_train, X_test, y_train, y_test = (
        load_theta_dataset(
            fold,
            topics
        )
    )

    # --------------------------------------------------------
    # Train classifier
    # --------------------------------------------------------

    clf = LogisticRegression(
        max_iter=LR_MAX_ITER,
        class_weight=LR_CLASS_WEIGHT,
        random_state=SEED
    )

    clf.fit(
        X_train,
        y_train
    )

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    y_pred = clf.predict(
        X_test
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_test,
            y_pred
        )
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    elapsed = (
        time.time()
        - start_time
    )

    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    results.append(
        {
            "ExperimentID":
                f"NG_F{fold}_K{topics}",

            "Fold":
                fold,

            "Topics":
                topics,

            "Accuracy":
                accuracy,

            "BalancedAccuracy":
                balanced_accuracy,

            "Precision":
                precision,

            "Recall":
                recall,

            "F1":
                f1,

            "MCC":
                mcc,

            "RuntimeSeconds":
                elapsed
        }
    )

    # --------------------------------------------------------
    # Save checkpoint after every experiment
    # --------------------------------------------------------

    pd.DataFrame(
        results
    ).to_csv(
        LR_CHECKPOINT_FILE,
        index=False
    )

    # --------------------------------------------------------
    # Memory cleanup
    # --------------------------------------------------------

    del (
        X_train,
        X_test,
        y_train,
        y_test,
        clf,
        y_pred
    )

    gc.collect()

# ============================================================
# Final result
# ============================================================

lr_results_df = (
    pd.DataFrame(results)
    .drop_duplicates(
        subset=[
            "Fold",
            "Topics"
        ],
        keep="last"
    )
    .sort_values(
        [
            "Fold",
            "Topics"
        ]
    )
    .reset_index(drop=True)
)

# ============================================================
# Integrity validation
# ============================================================

assert len(lr_results_df) == 495

assert (
    lr_results_df[
        ["Fold", "Topics"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    lr_results_df[
        [
            "Accuracy",
            "BalancedAccuracy",
            "Precision",
            "Recall",
            "F1",
            "MCC"
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

for fold in range(1, N_FOLDS + 1):

    fold_topics = (
        lr_results_df.loc[
            lr_results_df["Fold"] == fold,
            "Topics"
        ]
        .tolist()
    )

    assert fold_topics == list(
        range(2, 101)
    )

# ============================================================
# Save final results
# ============================================================

lr_results_df.to_csv(
    LR_FINAL_FILE,
    index=False
)

print("\n" + "=" * 80)
print("LOGISTIC REGRESSION COMPLETE")
print("=" * 80)

print(
    "\nRows              :",
    len(lr_results_df)
)

print(
    "Unique Fold-K     :",
    lr_results_df[
        ["Fold", "Topics"]
    ].drop_duplicates().shape[0]
)

print(
    "Missing values    :",
    lr_results_df.isna().sum().sum()
)

print(
    "Mean Accuracy     :",
    lr_results_df["Accuracy"].mean()
)

print(
    "Mean Macro-F1     :",
    lr_results_df["F1"].mean()
)

print(
    "Mean MCC          :",
    lr_results_df["MCC"].mean()
)

# ============================================================
# Best K by MCC within each fold
# ============================================================

best_mcc_rows = (
    lr_results_df.loc[
        lr_results_df
        .groupby("Fold")["MCC"]
        .idxmax()
    ][
        [
            "Fold",
            "Topics",
            "Accuracy",
            "BalancedAccuracy",
            "F1",
            "MCC"
        ]
    ]
    .rename(
        columns={
            "Topics":
                "BestK"
        }
    )
    .sort_values("Fold")
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("BEST K BY LOGISTIC-REGRESSION MCC")
print("=" * 80)

display(
    best_mcc_rows
)

print("\nSaved:")
print(LR_FINAL_FILE)

print("\n" + "=" * 80)
print("✓ 495 LOGISTIC-REGRESSION EXPERIMENTS VERIFIED")
print("✓ ALL FOLDS CONTAIN K = 2 ... 100")
print("✓ NO MISSING PERFORMANCE VALUES")
print("=" * 80)

NG05-3 — LOGISTIC REGRESSION: ALL 495 EXPERIMENTS

Logistic Regression Configuration
max_iter     : 1000
class_weight : balanced
random_seed  : 42

No checkpoint found.
Starting from the beginning.
Remaining experiments: 495


Logistic Regression:   0%|          | 0/495 [00:00<?, ?it/s]


LOGISTIC REGRESSION COMPLETE

Rows              : 495
Unique Fold-K     : 495
Missing values    : 0
Mean Accuracy     : 0.48636841249137897
Mean Macro-F1     : 0.47046499698826433
Mean MCC          : 0.4595588755581924

BEST K BY LOGISTIC-REGRESSION MCC


,Fold,BestK,Accuracy,BalancedAccuracy,F1,MCC
0,1,97,0.559016,0.549151,0.549288,0.535829
1,2,90,0.556831,0.546354,0.544831,0.533701
2,3,79,0.574474,0.562472,0.560293,0.552060
3,4,87,0.567641,0.557496,0.553888,0.544870
4,5,92,0.572561,0.562609,0.558744,0.550140



Saved:
/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/classification/ng05_logistic_regression_all_models.csv

✓ 495 LOGISTIC-REGRESSION EXPERIMENTS VERIFIED
✓ ALL FOLDS CONTAIN K = 2 ... 100
✓ NO MISSING PERFORMANCE VALUES


In [ ]:

# ============================================================
# NG05-4 — Single Random Forest Validation
#          Original TopicEvalBench RF Configuration
# ============================================================

import time

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    matthews_corrcoef
)

print("=" * 80)
print("NG05-4 — SINGLE RANDOM FOREST VALIDATION")
print("=" * 80)

# ============================================================
# Original Random Forest configuration
# ============================================================

RF_N_ESTIMATORS = 500
RF_MAX_DEPTH = None
RF_MIN_SAMPLES_SPLIT = 2
RF_MIN_SAMPLES_LEAF = 1
RF_MAX_FEATURES = "sqrt"
RF_CLASS_WEIGHT = "balanced"
RF_N_JOBS = -1

print("\nRandom Forest Configuration")
print("n_estimators      :", RF_N_ESTIMATORS)
print("max_depth         :", RF_MAX_DEPTH)
print("min_samples_split :", RF_MIN_SAMPLES_SPLIT)
print("min_samples_leaf  :", RF_MIN_SAMPLES_LEAF)
print("max_features      :", RF_MAX_FEATURES)
print("class_weight      :", RF_CLASS_WEIGHT)
print("random_seed       :", SEED)
print("n_jobs            :", RF_N_JOBS)

# ============================================================
# Representative experiment
# ============================================================

TEST_FOLD = 1
TEST_K = 2

X_train, X_test, y_train, y_test = (
    load_theta_dataset(
        TEST_FOLD,
        TEST_K
    )
)

print("\nTrain shape :", X_train.shape)
print("Test shape  :", X_test.shape)

# ============================================================
# Train
# ============================================================

start_time = time.time()

rf = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_split=RF_MIN_SAMPLES_SPLIT,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    max_features=RF_MAX_FEATURES,
    class_weight=RF_CLASS_WEIGHT,
    random_state=SEED,
    n_jobs=RF_N_JOBS
)

rf.fit(
    X_train,
    y_train
)

y_pred = rf.predict(
    X_test
)

runtime = time.time() - start_time

# ============================================================
# Metrics
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )
)

mcc = matthews_corrcoef(
    y_test,
    y_pred
)

single_rf_result = pd.DataFrame(
    [{
        "ExperimentID":
            f"NG_F{TEST_FOLD}_K{TEST_K}",

        "Fold":
            TEST_FOLD,

        "Topics":
            TEST_K,

        "Accuracy":
            accuracy,

        "BalancedAccuracy":
            balanced_accuracy,

        "Precision":
            precision,

        "Recall":
            recall,

        "F1":
            f1,

        "MCC":
            mcc,

        "RuntimeSeconds":
            runtime
    }]
)

print("\n" + "=" * 80)
print("SINGLE-EXPERIMENT RANDOM FOREST RESULT")
print("=" * 80)

display(single_rf_result)

# ============================================================
# Validation
# ============================================================

assert np.isfinite(
    single_rf_result[
        [
            "Accuracy",
            "BalancedAccuracy",
            "Precision",
            "Recall",
            "F1",
            "MCC"
        ]
    ].to_numpy()
).all()

assert 0 <= accuracy <= 1
assert 0 <= balanced_accuracy <= 1
assert 0 <= precision <= 1
assert 0 <= recall <= 1
assert 0 <= f1 <= 1
assert -1 <= mcc <= 1

print("\nRuntime :", round(runtime, 2), "seconds")

print("\n" + "=" * 80)
print("✓ RANDOM FOREST TRAINED SUCCESSFULLY")
print("✓ PREDICTIONS GENERATED SUCCESSFULLY")
print("✓ ALL PERFORMANCE METRICS VALID")
print("=" * 80)

NG05-4 — SINGLE RANDOM FOREST VALIDATION

Random Forest Configuration
n_estimators      : 500
max_depth         : None
min_samples_split : 2
min_samples_leaf  : 1
max_features      : sqrt
class_weight      : balanced
random_seed       : 42
n_jobs            : -1

Train shape : (14637, 2)
Test shape  : (3660, 2)

SINGLE-EXPERIMENT RANDOM FOREST RESULT


,ExperimentID,Fold,Topics,Accuracy,BalancedAccuracy,Precision,Recall,F1,MCC,RuntimeSeconds
0,NG_F1_K2,1,2,0.066667,0.065437,0.065274,0.065437,0.065268,0.017104,29.706799



Runtime : 29.71 seconds

✓ RANDOM FOREST TRAINED SUCCESSFULLY
✓ PREDICTIONS GENERATED SUCCESSFULLY
✓ ALL PERFORMANCE METRICS VALID


In [ ]:

# ============================================================
# NG05-5 — Random Forest over All 495 Experiments
#          Checkpoint / Resume Support
# ============================================================

import gc
import time
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    matthews_corrcoef
)

print("=" * 80)
print("NG05-5 — RANDOM FOREST: ALL 495 EXPERIMENTS")
print("=" * 80)

# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

RF_CHECKPOINT_FILE = (
    CLASSIFICATION_DIR / "ng05_random_forest_checkpoint.csv"
)

RF_FINAL_FILE = (
    CLASSIFICATION_DIR / "ng05_random_forest_all_models.csv"
)

# ------------------------------------------------------------
# Configuration from NG05-4
# ------------------------------------------------------------

RF_N_ESTIMATORS = 500
RF_MAX_DEPTH = None
RF_MIN_SAMPLES_SPLIT = 2
RF_MIN_SAMPLES_LEAF = 1
RF_MAX_FEATURES = "sqrt"
RF_CLASS_WEIGHT = "balanced"
RF_N_JOBS = -1

print("\nRandom Forest Configuration")
print("n_estimators      :", RF_N_ESTIMATORS)
print("max_depth         :", RF_MAX_DEPTH)
print("min_samples_split :", RF_MIN_SAMPLES_SPLIT)
print("min_samples_leaf  :", RF_MIN_SAMPLES_LEAF)
print("max_features      :", RF_MAX_FEATURES)
print("class_weight      :", RF_CLASS_WEIGHT)
print("random_seed       :", SEED)
print("n_jobs            :", RF_N_JOBS)

# ------------------------------------------------------------
# Resume checkpoint
# ------------------------------------------------------------

if RF_CHECKPOINT_FILE.exists():

    checkpoint_df = pd.read_csv(RF_CHECKPOINT_FILE)

    results = checkpoint_df.to_dict("records")

    completed = set(
        zip(
            checkpoint_df["Fold"].astype(int),
            checkpoint_df["Topics"].astype(int)
        )
    )

    print("\nCheckpoint found.")
    print("Completed experiments:", len(completed))

else:

    results = []
    completed = set()

    print("\nNo checkpoint found.")
    print("Starting from the beginning.")

# ------------------------------------------------------------
# Generate 495 Fold-K experiments
# ------------------------------------------------------------

experiments = []

for fold in range(1, N_FOLDS + 1):
    for topics in range(2, 101):
        experiments.append((fold, topics))

assert len(experiments) == 495

remaining = [
    x for x in experiments
    if x not in completed
]

print("Remaining experiments:", len(remaining))

# ------------------------------------------------------------
# Evaluate all remaining experiments
# ------------------------------------------------------------

for fold, topics in tqdm(
    remaining,
    desc="Random Forest"
):

    start_time = time.time()

    # Load topic-vector dataset
    X_train, X_test, y_train, y_test = load_theta_dataset(
        fold,
        topics
    )

    # Train Random Forest
    clf = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_split=RF_MIN_SAMPLES_SPLIT,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        max_features=RF_MAX_FEATURES,
        class_weight=RF_CLASS_WEIGHT,
        random_state=SEED,
        n_jobs=RF_N_JOBS
    )

    clf.fit(X_train, y_train)

    # Prediction
    y_pred = clf.predict(X_test)

    # Performance metrics
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        y_pred
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    elapsed = time.time() - start_time

    # Store result
    result = {
        "ExperimentID": f"NG_F{fold}_K{topics}",
        "Fold": fold,
        "Topics": topics,
        "Accuracy": accuracy,
        "BalancedAccuracy": balanced_accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "MCC": mcc,
        "RuntimeSeconds": elapsed
    }

    results.append(result)

    # Save checkpoint after every model
    pd.DataFrame(results).to_csv(
        RF_CHECKPOINT_FILE,
        index=False
    )

    # Cleanup
    del X_train
    del X_test
    del y_train
    del y_test
    del clf
    del y_pred

    gc.collect()

# ------------------------------------------------------------
# Create final dataframe
# ------------------------------------------------------------

rf_results_df = pd.DataFrame(results)

rf_results_df = (
    rf_results_df
    .drop_duplicates(
        subset=["Fold", "Topics"],
        keep="last"
    )
    .sort_values(["Fold", "Topics"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(rf_results_df) == 495

assert (
    rf_results_df[
        ["Fold", "Topics"]
    ].duplicated().sum()
    == 0
)

metric_columns = [
    "Accuracy",
    "BalancedAccuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC"
]

assert (
    rf_results_df[
        metric_columns
    ].isna().sum().sum()
    == 0
)

assert np.isfinite(
    rf_results_df[
        metric_columns
    ].to_numpy()
).all()

for fold in range(1, N_FOLDS + 1):

    fold_topics = rf_results_df.loc[
        rf_results_df["Fold"] == fold,
        "Topics"
    ].tolist()

    assert fold_topics == list(range(2, 101))

# ------------------------------------------------------------
# Save final results
# ------------------------------------------------------------

rf_results_df.to_csv(
    RF_FINAL_FILE,
    index=False
)

# ------------------------------------------------------------
# Overall summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RANDOM FOREST COMPLETE")
print("=" * 80)

print("Rows              :", len(rf_results_df))
print(
    "Unique Fold-K     :",
    rf_results_df[
        ["Fold", "Topics"]
    ].drop_duplicates().shape[0]
)
print(
    "Missing values    :",
    rf_results_df[
        metric_columns
    ].isna().sum().sum()
)
print(
    "Mean Accuracy     :",
    rf_results_df["Accuracy"].mean()
)
print(
    "Mean Macro-F1     :",
    rf_results_df["F1"].mean()
)
print(
    "Mean MCC          :",
    rf_results_df["MCC"].mean()
)
print(
    "Mean runtime/model:",
    rf_results_df["RuntimeSeconds"].mean()
)

# ------------------------------------------------------------
# Best K by MCC for each fold
# ------------------------------------------------------------

best_indices = (
    rf_results_df
    .groupby("Fold")["MCC"]
    .idxmax()
)

best_rf_mcc_rows = rf_results_df.loc[
    best_indices,
    [
        "Fold",
        "Topics",
        "Accuracy",
        "BalancedAccuracy",
        "F1",
        "MCC"
    ]
].copy()

best_rf_mcc_rows = (
    best_rf_mcc_rows
    .rename(columns={"Topics": "BestK"})
    .sort_values("Fold")
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("BEST K BY RANDOM-FOREST MCC")
print("=" * 80)

display(best_rf_mcc_rows)

print("\nSaved:")
print(RF_FINAL_FILE)

print("\n" + "=" * 80)
print("✓ 495 RANDOM-FOREST EXPERIMENTS VERIFIED")
print("✓ ALL FOLDS CONTAIN K = 2 ... 100")
print("✓ NO MISSING PERFORMANCE VALUES")
print("=" * 80)

NG05-5 — RANDOM FOREST: ALL 495 EXPERIMENTS

Random Forest Configuration
n_estimators      : 500
max_depth         : None
min_samples_split : 2
min_samples_leaf  : 1
max_features      : sqrt
class_weight      : balanced
random_seed       : 42
n_jobs            : -1

Checkpoint found.
Completed experiments: 372
Remaining experiments: 123


Random Forest:   0%|          | 0/123 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# NG05-6 — Single RBF-SVM Validation
#          Original TopicEvalBench SVM Configuration
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    matthews_corrcoef
)

print("=" * 80)
print("NG05-6 — SINGLE RBF-SVM VALIDATION")
print("=" * 80)

# ============================================================
# Original SVM configuration
# ============================================================

SVM_KERNEL = "rbf"
SVM_C = 1.0
SVM_GAMMA = "scale"
SVM_CLASS_WEIGHT = "balanced"

print("\nSVM Configuration")
print("kernel       :", SVM_KERNEL)
print("C            :", SVM_C)
print("gamma        :", SVM_GAMMA)
print("class_weight :", SVM_CLASS_WEIGHT)
print("random_seed  :", SEED)

# ============================================================
# Same validation experiment used previously
# ============================================================

TEST_FOLD = 1
TEST_K = 2

X_train, X_test, y_train, y_test = load_theta_dataset(
    TEST_FOLD,
    TEST_K
)

print("\nExperiment  : Fold", TEST_FOLD, ", K =", TEST_K)
print("Train shape :", X_train.shape)
print("Test shape  :", X_test.shape)
print("Classes     :", len(np.unique(y_train)))

# ============================================================
# Train RBF-SVM
# ============================================================

start_time = time.time()

svm = SVC(
    kernel=SVM_KERNEL,
    C=SVM_C,
    gamma=SVM_GAMMA,
    class_weight=SVM_CLASS_WEIGHT,
    random_state=SEED
)

svm.fit(
    X_train,
    y_train
)

# ============================================================
# Prediction
# ============================================================

y_pred = svm.predict(
    X_test
)

runtime = time.time() - start_time

# ============================================================
# Performance metrics
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )
)

mcc = matthews_corrcoef(
    y_test,
    y_pred
)

# ============================================================
# Result
# ============================================================

single_svm_result = pd.DataFrame(
    [{
        "ExperimentID": f"NG_F{TEST_FOLD}_K{TEST_K}",
        "Fold": TEST_FOLD,
        "Topics": TEST_K,
        "Accuracy": accuracy,
        "BalancedAccuracy": balanced_accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "MCC": mcc,
        "RuntimeSeconds": runtime
    }]
)

print("\n" + "=" * 80)
print("SINGLE-EXPERIMENT RBF-SVM RESULT")
print("=" * 80)

display(single_svm_result)

# ============================================================
# Validation
# ============================================================

metric_columns = [
    "Accuracy",
    "BalancedAccuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC"
]

assert np.isfinite(
    single_svm_result[
        metric_columns
    ].to_numpy()
).all()

assert 0 <= accuracy <= 1
assert 0 <= balanced_accuracy <= 1
assert 0 <= precision <= 1
assert 0 <= recall <= 1
assert 0 <= f1 <= 1
assert -1 <= mcc <= 1

print("\nRuntime :", round(runtime, 2), "seconds")

print("\n" + "=" * 80)
print("✓ RBF-SVM TRAINED SUCCESSFULLY")
print("✓ PREDICTIONS GENERATED SUCCESSFULLY")
print("✓ ALL DOWNSTREAM METRICS VALID")
print("=" * 80)

NG05-6 — SINGLE RBF-SVM VALIDATION

SVM Configuration
kernel       : rbf
C            : 1.0
gamma        : scale
class_weight : balanced
random_seed  : 42

Experiment  : Fold 1 , K = 2
Train shape : (14637, 2)
Test shape  : (3660, 2)
Classes     : 20

SINGLE-EXPERIMENT RBF-SVM RESULT


,ExperimentID,Fold,Topics,Accuracy,BalancedAccuracy,Precision,Recall,F1,MCC,RuntimeSeconds
0,NG_F1_K2,1,2,0.059836,0.056269,0.038104,0.056269,0.01766,0.012045,36.086523



Runtime : 36.09 seconds

✓ RBF-SVM TRAINED SUCCESSFULLY
✓ PREDICTIONS GENERATED SUCCESSFULLY
✓ ALL DOWNSTREAM METRICS VALID


In [ ]:
# ============================================================
# NG05-7 — RBF-SVM over All 495 Experiments
#          Checkpoint / Resume Support
# ============================================================

import gc
import time
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    matthews_corrcoef
)

print("=" * 80)
print("NG05-7 — RBF-SVM: ALL 495 EXPERIMENTS")
print("=" * 80)

# ============================================================
# Output files
# ============================================================

SVM_CHECKPOINT_FILE = (
    CLASSIFICATION_DIR
    / "ng05_svm_checkpoint.csv"
)

SVM_FINAL_FILE = (
    CLASSIFICATION_DIR
    / "ng05_svm_all_models.csv"
)

# ============================================================
# Original SVM configuration
# ============================================================

SVM_KERNEL = "rbf"
SVM_C = 1.0
SVM_GAMMA = "scale"
SVM_CLASS_WEIGHT = "balanced"

print("\nSVM Configuration")
print("kernel       :", SVM_KERNEL)
print("C            :", SVM_C)
print("gamma        :", SVM_GAMMA)
print("class_weight :", SVM_CLASS_WEIGHT)
print("random_seed  :", SEED)

# ============================================================
# Resume checkpoint
# ============================================================

if SVM_CHECKPOINT_FILE.exists():

    checkpoint_df = pd.read_csv(
        SVM_CHECKPOINT_FILE
    )

    results = checkpoint_df.to_dict(
        "records"
    )

    completed = set(
        zip(
            checkpoint_df["Fold"].astype(int),
            checkpoint_df["Topics"].astype(int)
        )
    )

    print("\nCheckpoint found.")
    print("Completed experiments:", len(completed))

else:

    results = []
    completed = set()

    print("\nNo checkpoint found.")
    print("Starting from the beginning.")

# ============================================================
# Construct all 495 experiments
# ============================================================

experiments = []

for fold in range(1, N_FOLDS + 1):
    for topics in range(2, 101):
        experiments.append(
            (fold, topics)
        )

assert len(experiments) == 495

remaining = [
    experiment
    for experiment in experiments
    if experiment not in completed
]

print(
    "Remaining experiments:",
    len(remaining)
)

# ============================================================
# Run experiments
# ============================================================

for fold, topics in tqdm(
    remaining,
    desc="RBF-SVM"
):

    start_time = time.time()

    # --------------------------------------------------------
    # Load topic-vector features
    # --------------------------------------------------------

    X_train, X_test, y_train, y_test = (
        load_theta_dataset(
            fold,
            topics
        )
    )

    # --------------------------------------------------------
    # Train SVM
    # --------------------------------------------------------

    clf = SVC(
        kernel=SVM_KERNEL,
        C=SVM_C,
        gamma=SVM_GAMMA,
        class_weight=SVM_CLASS_WEIGHT,
        random_state=SEED
    )

    clf.fit(
        X_train,
        y_train
    )

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    y_pred = clf.predict(
        X_test
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_test,
            y_pred
        )
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    elapsed = time.time() - start_time

    # --------------------------------------------------------
    # Save result
    # --------------------------------------------------------

    results.append(
        {
            "ExperimentID":
                f"NG_F{fold}_K{topics}",

            "Fold":
                fold,

            "Topics":
                topics,

            "Accuracy":
                accuracy,

            "BalancedAccuracy":
                balanced_accuracy,

            "Precision":
                precision,

            "Recall":
                recall,

            "F1":
                f1,

            "MCC":
                mcc,

            "RuntimeSeconds":
                elapsed
        }
    )

    # --------------------------------------------------------
    # Checkpoint after EVERY experiment
    # --------------------------------------------------------

    pd.DataFrame(
        results
    ).to_csv(
        SVM_CHECKPOINT_FILE,
        index=False
    )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del X_train
    del X_test
    del y_train
    del y_test
    del clf
    del y_pred

    gc.collect()

# ============================================================
# Final dataframe
# ============================================================

svm_results_df = pd.DataFrame(
    results
)

svm_results_df = (
    svm_results_df
    .drop_duplicates(
        subset=[
            "Fold",
            "Topics"
        ],
        keep="last"
    )
    .sort_values(
        [
            "Fold",
            "Topics"
        ]
    )
    .reset_index(drop=True)
)

# ============================================================
# Integrity validation
# ============================================================

assert len(svm_results_df) == 495

assert (
    svm_results_df[
        ["Fold", "Topics"]
    ]
    .duplicated()
    .sum()
    == 0
)

metric_columns = [
    "Accuracy",
    "BalancedAccuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC"
]

assert (
    svm_results_df[
        metric_columns
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

assert np.isfinite(
    svm_results_df[
        metric_columns
    ].to_numpy()
).all()

for fold in range(
    1,
    N_FOLDS + 1
):

    fold_topics = (
        svm_results_df.loc[
            svm_results_df["Fold"] == fold,
            "Topics"
        ]
        .tolist()
    )

    assert fold_topics == list(
        range(2, 101)
    )

# ============================================================
# Save final result
# ============================================================

svm_results_df.to_csv(
    SVM_FINAL_FILE,
    index=False
)

# ============================================================
# Overall summary
# ============================================================

print("\n" + "=" * 80)
print("RBF-SVM COMPLETE")
print("=" * 80)

print(
    "Rows              :",
    len(svm_results_df)
)

print(
    "Unique Fold-K     :",
    svm_results_df[
        ["Fold", "Topics"]
    ].drop_duplicates().shape[0]
)

print(
    "Missing values    :",
    svm_results_df[
        metric_columns
    ].isna().sum().sum()
)

print(
    "Mean Accuracy     :",
    svm_results_df["Accuracy"].mean()
)

print(
    "Mean Macro-F1     :",
    svm_results_df["F1"].mean()
)

print(
    "Mean MCC          :",
    svm_results_df["MCC"].mean()
)

print(
    "Mean runtime/model:",
    svm_results_df[
        "RuntimeSeconds"
    ].mean()
)

# ============================================================
# Best K by MCC within each fold
# ============================================================

best_indices = (
    svm_results_df
    .groupby("Fold")["MCC"]
    .idxmax()
)

best_svm_mcc_rows = (
    svm_results_df.loc[
        best_indices,
        [
            "Fold",
            "Topics",
            "Accuracy",
            "BalancedAccuracy",
            "F1",
            "MCC"
        ]
    ]
    .copy()
)

best_svm_mcc_rows = (
    best_svm_mcc_rows
    .rename(
        columns={
            "Topics": "BestK"
        }
    )
    .sort_values("Fold")
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("BEST K BY RBF-SVM MCC")
print("=" * 80)

display(
    best_svm_mcc_rows
)

print("\nSaved:")
print(SVM_FINAL_FILE)

print("\n" + "=" * 80)
print("✓ 495 RBF-SVM EXPERIMENTS VERIFIED")
print("✓ ALL FOLDS CONTAIN K = 2 ... 100")
print("✓ NO MISSING PERFORMANCE VALUES")
print("=" * 80)

NG05-7 — RBF-SVM: ALL 495 EXPERIMENTS

SVM Configuration
kernel       : rbf
C            : 1.0
gamma        : scale
class_weight : balanced
random_seed  : 42

No checkpoint found.
Starting from the beginning.
Remaining experiments: 495


RBF-SVM:   0%|          | 0/495 [00:00<?, ?it/s]


RBF-SVM COMPLETE
Rows              : 495
Unique Fold-K     : 495
Missing values    : 0
Mean Accuracy     : 0.5003128721945114
Mean Macro-F1     : 0.4842588011022845
Mean MCC          : 0.4743841699843297
Mean runtime/model: 20.443638583867237

BEST K BY RBF-SVM MCC


,Fold,BestK,Accuracy,BalancedAccuracy,F1,MCC
0,1,97,0.568306,0.558033,0.559216,0.545642
1,2,86,0.571585,0.559945,0.557438,0.549076
2,3,79,0.590599,0.578271,0.576495,0.569071
3,4,99,0.586226,0.574883,0.571846,0.564462
4,5,95,0.586499,0.575971,0.575585,0.564888



Saved:
/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/classification/ng05_svm_all_models.csv

✓ 495 RBF-SVM EXPERIMENTS VERIFIED
✓ ALL FOLDS CONTAIN K = 2 ... 100
✓ NO MISSING PERFORMANCE VALUES


In [ ]:
# ============================================================
# NG05-8 — Consolidate Classification Results
#          LR + RF + RBF-SVM
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 80)
print("NG05-8 — CONSOLIDATE DOWNSTREAM CLASSIFICATION RESULTS")
print("=" * 80)

# ============================================================
# Input files
# ============================================================

LR_FILE = (
    CLASSIFICATION_DIR
    / "ng05_logistic_regression_all_models.csv"
)

RF_FILE = (
    CLASSIFICATION_DIR
    / "ng05_random_forest_all_models.csv"
)

SVM_FILE = (
    CLASSIFICATION_DIR
    / "ng05_svm_all_models.csv"
)

assert LR_FILE.exists(), f"Missing: {LR_FILE}"
assert RF_FILE.exists(), f"Missing: {RF_FILE}"
assert SVM_FILE.exists(), f"Missing: {SVM_FILE}"

# ============================================================
# Load
# ============================================================

lr = pd.read_csv(LR_FILE)
rf = pd.read_csv(RF_FILE)
svm = pd.read_csv(SVM_FILE)

print("\nInput rows")
print("Logistic Regression :", len(lr))
print("Random Forest       :", len(rf))
print("RBF-SVM             :", len(svm))

# ============================================================
# Structural validation
# ============================================================

for name, df in [
    ("LR", lr),
    ("RF", rf),
    ("SVM", svm)
]:

    assert len(df) == 495

    assert (
        df[["Fold", "Topics"]]
        .duplicated()
        .sum()
        == 0
    )

    assert df["MCC"].notna().all()

    assert np.isfinite(
        df["MCC"].to_numpy()
    ).all()

print("\n✓ All three classifier result files validated.")

# ============================================================
# Keep performance columns and rename by classifier
# ============================================================

lr_part = lr[
    [
        "Fold",
        "Topics",
        "Accuracy",
        "BalancedAccuracy",
        "Precision",
        "Recall",
        "F1",
        "MCC"
    ]
].copy()

lr_part = lr_part.rename(
    columns={
        "Accuracy": "LR_Accuracy",
        "BalancedAccuracy": "LR_BalancedAccuracy",
        "Precision": "LR_Precision",
        "Recall": "LR_Recall",
        "F1": "LR_F1",
        "MCC": "LR_MCC"
    }
)

rf_part = rf[
    [
        "Fold",
        "Topics",
        "Accuracy",
        "BalancedAccuracy",
        "Precision",
        "Recall",
        "F1",
        "MCC"
    ]
].copy()

rf_part = rf_part.rename(
    columns={
        "Accuracy": "RF_Accuracy",
        "BalancedAccuracy": "RF_BalancedAccuracy",
        "Precision": "RF_Precision",
        "Recall": "RF_Recall",
        "F1": "RF_F1",
        "MCC": "RF_MCC"
    }
)

svm_part = svm[
    [
        "Fold",
        "Topics",
        "Accuracy",
        "BalancedAccuracy",
        "Precision",
        "Recall",
        "F1",
        "MCC"
    ]
].copy()

svm_part = svm_part.rename(
    columns={
        "Accuracy": "SVM_Accuracy",
        "BalancedAccuracy": "SVM_BalancedAccuracy",
        "Precision": "SVM_Precision",
        "Recall": "SVM_Recall",
        "F1": "SVM_F1",
        "MCC": "SVM_MCC"
    }
)

# ============================================================
# Merge
# ============================================================

classification_master = (
    lr_part
    .merge(
        rf_part,
        on=["Fold", "Topics"],
        how="inner",
        validate="one_to_one"
    )
    .merge(
        svm_part,
        on=["Fold", "Topics"],
        how="inner",
        validate="one_to_one"
    )
    .sort_values(
        ["Fold", "Topics"]
    )
    .reset_index(drop=True)
)

assert len(classification_master) == 495

# ============================================================
# Mean downstream performance across classifiers
# ============================================================

classification_master["MeanMCC"] = (
    classification_master[
        [
            "LR_MCC",
            "RF_MCC",
            "SVM_MCC"
        ]
    ]
    .mean(axis=1)
)

classification_master["MeanAccuracy"] = (
    classification_master[
        [
            "LR_Accuracy",
            "RF_Accuracy",
            "SVM_Accuracy"
        ]
    ]
    .mean(axis=1)
)

classification_master["MeanF1"] = (
    classification_master[
        [
            "LR_F1",
            "RF_F1",
            "SVM_F1"
        ]
    ]
    .mean(axis=1)
)

# ============================================================
# Final integrity checks
# ============================================================

assert (
    classification_master[
        ["Fold", "Topics"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert classification_master["MeanMCC"].notna().all()

assert np.isfinite(
    classification_master["MeanMCC"]
    .to_numpy()
).all()

# ============================================================
# Save master classification table
# ============================================================

CLASSIFICATION_MASTER_FILE = (
    CLASSIFICATION_DIR
    / "ng05_classification_master.csv"
)

classification_master.to_csv(
    CLASSIFICATION_MASTER_FILE,
    index=False
)

# ============================================================
# Best K separately for each classifier + mean MCC
# ============================================================

summary_rows = []

for fold in range(1, N_FOLDS + 1):

    fold_df = classification_master[
        classification_master["Fold"] == fold
    ]

    lr_row = fold_df.loc[
        fold_df["LR_MCC"].idxmax()
    ]

    rf_row = fold_df.loc[
        fold_df["RF_MCC"].idxmax()
    ]

    svm_row = fold_df.loc[
        fold_df["SVM_MCC"].idxmax()
    ]

    mean_row = fold_df.loc[
        fold_df["MeanMCC"].idxmax()
    ]

    summary_rows.append(
        {
            "Fold": fold,

            "LR_BestK":
                int(lr_row["Topics"]),

            "LR_BestMCC":
                lr_row["LR_MCC"],

            "RF_BestK":
                int(rf_row["Topics"]),

            "RF_BestMCC":
                rf_row["RF_MCC"],

            "SVM_BestK":
                int(svm_row["Topics"]),

            "SVM_BestMCC":
                svm_row["SVM_MCC"],

            "MeanMCC_BestK":
                int(mean_row["Topics"]),

            "BestMeanMCC":
                mean_row["MeanMCC"]
        }
    )

downstream_summary = pd.DataFrame(
    summary_rows
)

# ============================================================
# Save summary
# ============================================================

DOWNSTREAM_SUMMARY_FILE = (
    CLASSIFICATION_DIR
    / "ng05_downstream_optimal_k_summary.csv"
)

downstream_summary.to_csv(
    DOWNSTREAM_SUMMARY_FILE,
    index=False
)

# ============================================================
# Display
# ============================================================

print("\n" + "=" * 80)
print("DOWNSTREAM-OPTIMAL K BY MCC")
print("=" * 80)

display(
    downstream_summary
)

print("\nOverall mean MCC")
print(
    "LR   :",
    classification_master["LR_MCC"].mean()
)

print(
    "RF   :",
    classification_master["RF_MCC"].mean()
)

print(
    "SVM  :",
    classification_master["SVM_MCC"].mean()
)

print(
    "Mean :",
    classification_master["MeanMCC"].mean()
)

print("\nSaved:")
print(CLASSIFICATION_MASTER_FILE)
print(DOWNSTREAM_SUMMARY_FILE)

print("\n" + "=" * 80)
print("✓ 495 FOLD-K EXPERIMENTS CONSOLIDATED")
print("✓ LR + RF + SVM ALIGNED ONE-TO-ONE")
print("✓ DOWNSTREAM MEAN MCC COMPUTED")
print("=" * 80)

NG05-8 — CONSOLIDATE DOWNSTREAM CLASSIFICATION RESULTS

Input rows
Logistic Regression : 495
Random Forest       : 495
RBF-SVM             : 495

✓ All three classifier result files validated.

DOWNSTREAM-OPTIMAL K BY MCC


,Fold,LR_BestK,LR_BestMCC,RF_BestK,RF_BestMCC,SVM_BestK,SVM_BestMCC,MeanMCC_BestK,BestMeanMCC
0,1,97,0.535829,97,0.522184,97,0.545642,97,0.534552
1,2,90,0.533701,95,0.531515,86,0.549076,95,0.534781
2,3,79,0.552060,79,0.543855,79,0.569071,79,0.554995
3,4,87,0.544870,99,0.533351,99,0.564462,99,0.546876
4,5,92,0.550140,81,0.537679,95,0.564888,81,0.547812



Overall mean MCC
LR   : 0.4595588755581924
RF   : 0.4536536201579641
SVM  : 0.4743841699843297
Mean : 0.4625322219001621

Saved:
/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/classification/ng05_classification_master.csv
/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/classification/ng05_downstream_optimal_k_summary.csv

✓ 495 FOLD-K EXPERIMENTS CONSOLIDATED
✓ LR + RF + SVM ALIGNED ONE-TO-ONE
✓ DOWNSTREAM MEAN MCC COMPUTED
